In [10]:
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition
import pyomo.environ as pyo
import pandas as pd

# ── Dados ──────────────────────────────────────────────────────────────────────
df = pd.DataFrame({
    'tempo': pd.to_datetime(list(range(0, 24)), unit='h',
                            origin='2026-03-13').strftime('%H:%M').tolist(),
    'P Demand': [
        1.9317, 1.6090, 1.4079, 1.3281, 1.3834, 1.6413,
        1.9395, 1.7383, 1.8341, 1.8354, 1.9312, 2.3645,
        2.2038, 2.2997, 2.1659, 2.5046, 2.7490, 4.0597,
        4.9924, 5.4257, 5.0491, 4.4294, 3.7692, 2.7716
    ],
    "P PV": [
        0.0000, 0.0000, 0.0000, 0.0000, 0.0796, 0.4565,
        1.0742, 1.5790, 2.4343, 2.7488, 3.5092, 3.8988,
        3.9734, 3.7105, 3.1671, 2.7282, 2.3926, 2.1764,
        1.9083, 1.4257, 0.0034, 0.0000, 0.0000, 0.0000
    ],
    "tariff": [
        0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
        0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
        0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.32629,
        0.51792, 0.51792, 0.51792, 0.32629, 0.22419, 0.22419
    ]
})

parameters = {
    "General": {"timestep": 60},
    "Grid":    {"Pmax": 100},
    "BESS": {
        "Pmax": 2.0,
        "eff": 0.90,
        "self_discharge": 0.05,
        "capacity": 7,
        "initial capacity": 0,
    }
}

df.set_index('tempo', inplace=True)


# ── Classes ────────────────────────────────────────────────────────────────────
class General:
    def __init__(self, parameters):
        self.timestep = parameters["General"]["timestep"]

class BESS:
    def __init__(self, parameters, general):
        self.general          = general
        self.Pmax             = parameters["BESS"]["Pmax"]
        self.eff              = parameters["BESS"]["eff"]
        self.initial_capacity = parameters["BESS"]["initial capacity"]
        self.capacity         = parameters["BESS"]["capacity"]
        self.self_discharge   = parameters["BESS"]["self_discharge"]

class Grid:
    def __init__(self, parameters, general):
        self.general = general
        self.Pmax    = parameters["Grid"]["Pmax"]

class PV:
    def __init__(self, parameters, general):
        self.parameters = parameters
        self.general = general

class SmartHome:
    def __init__(self, parameters, df):
        self.df      = df
        self.general = General(parameters)
        self.grid    = Grid(parameters, self.general)
        self.bess    = BESS(parameters, self.general)
        self.pv      = PV(parameters, self.general)

    def build(self):
        model = pyo.ConcreteModel('SmartHome')
        delta = self.general.timestep / 60  # 1.0 hora

        # ── Variáveis ──────────────────────────────────────────────────────────
        # Potência comprada (+) da rede
        model.Pgrid_buy = pyo.Var(self.df.index,
                              bounds=(0, self.grid.Pmax),
                              within=pyo.NonNegativeReals)

        # Potência vendida à rede (excedente PV)
        model.Pgrid_sell = pyo.Var(self.df.index,
                                   bounds=(0, self.grid.Pmax),
                                   within=pyo.NonNegativeReals)

        # Potência de carga da bateria (carregando)
        model.Pbess_charge = pyo.Var(self.df.index,
                                     bounds=(0, self.bess.Pmax),
                                     within=pyo.NonNegativeReals)

        # Potência de descarga da bateria (descarregando)
        model.Pbess_discharge = pyo.Var(self.df.index,
                                        bounds=(0, self.bess.Pmax),
                                        within=pyo.NonNegativeReals)

        # Energia armazenada na bateria
        model.E_bess = pyo.Var(self.df.index,
                               bounds=(0, self.bess.capacity),
                               within=pyo.Reals)

        # Estado da bateria: 1 = descarregando, 0 = carregando
        model.state = pyo.Var(self.df.index, within=pyo.Binary)

        # ── Restrições ─────────────────────────────────────────────────────────

        # 1) Balanço de potência em cada hora
        #    Pgrid_buy + PV + Pbess_discharge = Demanda + Pbess_charge + Pgrid_sell
        def power_balance_rule(model, t):
            return (+ model.Pgrid_buy[t]
                    + self.df.loc[t, 'P PV']
                    + model.Pbess_discharge[t]
                    ==
                    +  self.df.loc[t, 'P Demand']
                    + model.Pbess_charge[t]
                    + model.Pgrid_sell[t])

        model.power = pyo.Constraint(self.df.index, rule=power_balance_rule)

        # 2) Dinâmica da bateria hora a hora
        #    E[t] = E[t-1] + eff*delta*Pch[t] - delta*Pdis[t]/eff - beta*delta*E[t]
        def bess_energy_rule(model, t):
            idx = self.df.index.get_loc(t)
            eff  = self.bess.eff
            beta = self.bess.self_discharge

            charge_term    = eff  * delta * model.Pbess_charge[t]
            discharge_term = delta * model.Pbess_discharge[t] / eff
            loss_term      = beta * delta * model.E_bess[t]

            if idx == 0:
                E_prev = self.bess.initial_capacity  # hora 0: sem hora anterior
            else:
                t_prev = self.df.index[idx - 1]
                E_prev = model.E_bess[t_prev]

            return model.E_bess[t] == E_prev + charge_term - discharge_term - loss_term

        model.bess_energy = pyo.Constraint(self.df.index, rule=bess_energy_rule)

        # 3) Bateria só pode carregar OU descarregar (não os dois ao mesmo tempo)
    
        def bess_discharge_limit(model, t): # (só descarrega se state=1)
            return model.Pbess_discharge[t] <= self.bess.Pmax * model.state[t] 

        def bess_charge_limit(model, t): # (só carrega se state=0)
            return model.Pbess_charge[t] <= self.bess.Pmax * (1 - model.state[t])

        model.bess_dis_limit = pyo.Constraint(self.df.index, rule=bess_discharge_limit)
        model.bess_ch_limit  = pyo.Constraint(self.df.index, rule=bess_charge_limit)

        # ── Função objetivo ────────────────────────────────────────────────────
        # Minimizar custo total de compra de energia da rede
        def objective_rule(model):
            return delta * sum(self.df.loc[t, 'tariff'] * model.Pgrid_buy[t]
                               for t in self.df.index)

        model.funcao_objetivo = pyo.Objective(rule=objective_rule,
                                              sense=pyo.minimize)
        self.model = model

    def solve(self):
        solver   = SolverFactory('highs')
        solution = solver.solve(self.model, tee=True)  # tee=True imprime o log

        # Verifica se resolveu com sucesso
        if (solution.solver.status == SolverStatus.ok and
            solution.solver.termination_condition == TerminationCondition.optimal):
            print("\n✓ Solução ótima encontrada!")
            print(f"  Custo total: R$ {pyo.value(self.model.funcao_objetivo):.4f}")
            
            # Exibir valores das variáveis
            print("\n" + "="*80)
            print("VALORES DAS VARIÁVEIS")
            print("="*80)
            
            results = []
            for t in self.df.index:
                results.append({
                    'Hora': t,
                    'P_Grid_Buy (kW)': f"{pyo.value(self.model.Pgrid_buy[t]):.4f}",
                    'P_Grid_Sell (kW)': f"{pyo.value(self.model.Pgrid_sell[t]):.4f}",
                    'P_BESS_Charge (kW)': f"{pyo.value(self.model.Pbess_charge[t]):.4f}",
                    'P_BESS_Discharge (kW)': f"{pyo.value(self.model.Pbess_discharge[t]):.4f}",
                    'E_BESS (kWh)': f"{pyo.value(self.model.E_bess[t]):.4f}"
                })
            
            results_df = pd.DataFrame(results)
            print(results_df.to_string(index=False))


# ── Execução ───────────────────────────────────────────────────────────────────
smarthome = SmartHome(parameters, df)
smarthome.build()
smarthome.solve()


Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
MIP has 96 rows; 144 cols; 287 nonzeros; 24 integer variables (24 binary)
Coefficient ranges:
  Matrix  [9e-01, 2e+00]
  Cost    [2e-01, 5e-01]
  Bound   [1e+00, 1e+02]
  RHS     [2e-01, 5e+00]
Presolving model
96 rows, 138 cols, 281 nonzeros 0s
86 rows, 110 cols, 235 nonzeros 0s
69 rows, 93 cols, 184 nonzeros 0s
Presolve reductions: rows 69(-27); columns 93(-51); nonzeros 184(-103) 

Solving MIP model with:
   69 rows
   93 cols (23 binary, 0 integer, 0 implied int., 70 continuous, 0 domain fixed)
   184 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes   

In [ ]:
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition
import pyomo.environ as pyo
import pandas as pd

# ── Dados ──────────────────────────────────────────────────────────────────────
P_demand_data = [
    1.9317, 1.6090, 1.4079, 1.3281, 1.3834, 1.6413,
    1.9395, 1.7383, 1.8341, 1.8354, 1.9312, 2.3645,
    2.2038, 2.2997, 2.1659, 2.5046, 2.7490, 4.0597,
    4.9924, 5.4257, 5.0491, 4.4294, 3.7692, 2.7716
]

P_pv_data = [
    0.0000, 0.0000, 0.0000, 0.0000, 0.0796, 0.4565,
    1.0742, 1.5790, 2.4343, 2.7488, 3.5092, 3.8988,
    3.9734, 3.7105, 3.1671, 2.7282, 2.3926, 2.1764,
    1.9083, 1.4257, 0.0034, 0.0000, 0.0000, 0.0000
]

tariff_data = [
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.32629,
    0.51792, 0.51792, 0.51792, 0.32629, 0.22419, 0.22419
]

parameters = {
    "General": {"timestep": 60},
    "Grid":    {"Pmax": 100},
    "BESS": {
        "Pmax": 2.0,
        "eff": 0.90,
        "self_discharge": 0.05,
        "capacity": 7,
        "initial capacity": 0,
    }
}


# ── Classes ────────────────────────────────────────────────────────────────────
class General:
    def __init__(self, parameters):
        self.timestep = parameters["General"]["timestep"]

class BESS:
    def __init__(self, parameters, general):
        self.general          = general
        self.Pmax             = parameters["BESS"]["Pmax"]
        self.eff              = parameters["BESS"]["eff"]
        self.initial_capacity = parameters["BESS"]["initial capacity"]
        self.capacity         = parameters["BESS"]["capacity"]
        self.self_discharge   = parameters["BESS"]["self_discharge"]

class Grid:
    def __init__(self, parameters, general):
        self.general = general
        self.Pmax    = parameters["Grid"]["Pmax"]

class PV:
    def __init__(self, parameters, general):
        self.parameters = parameters
        self.general = general

class SmartHome:
    def __init__(self, parameters, P_demand_data, P_pv_data, tariff_data):
        self.P_demand_data = P_demand_data
        self.P_pv_data = P_pv_data
        self.tariff_data = tariff_data
        self.general = General(parameters)
        self.grid    = Grid(parameters, self.general)
        self.bess    = BESS(parameters, self.general)
        self.pv      = PV(parameters, self.general)

    def build(self):
        model = pyo.ConcreteModel('SmartHome')
        delta = self.general.timestep / 60  # 1.0 hora

        # ── Índices ────────────────────────────────────────────────────────────
        # Conjunto de horas (0 a 23)
        model.T = pyo.RangeSet(0, len(self.P_demand_data) - 1)

        # ── Parâmetros ────────────────────────────────────────────────────────────
        # Demanda de potência
        model.P_demand = pyo.Param(model.T,
                                   initialize=dict(enumerate(self.P_demand_data)))

        # Geração solar (PV)
        model.P_pv = pyo.Param(model.T,
                               initialize=dict(enumerate(self.P_pv_data)))

        # Tarifa de eletricidade
        model.tariff = pyo.Param(model.T,
                                 initialize=dict(enumerate(self.tariff_data)))

        # ── Variáveis ──────────────────────────────────────────────────────────────
        # Potência comprada (+) da rede
        model.Pgrid_buy = pyo.Var(model.T,
                              bounds=(0, self.grid.Pmax),
                              within=pyo.NonNegativeReals)

        # Potência vendida à rede (excedente PV)
        model.Pgrid_sell = pyo.Var(model.T,
                                   bounds=(0, self.grid.Pmax),
                                   within=pyo.NonNegativeReals)

        # Potência de carga da bateria (carregando)
        model.Pbess_charge = pyo.Var(model.T,
                                     bounds=(0, self.bess.Pmax),
                                     within=pyo.NonNegativeReals)

        # Potência de descarga da bateria (descarregando)
        model.Pbess_discharge = pyo.Var(model.T,
                                        bounds=(0, self.bess.Pmax),
                                        within=pyo.NonNegativeReals)

        # Energia armazenada na bateria
        model.E_bess = pyo.Var(model.T,
                               bounds=(0, self.bess.capacity),
                               within=pyo.Reals)

        # Estado da bateria: 1 = descarregando, 0 = carregando
        model.state = pyo.Var(model.T, within=pyo.Binary)

        # ── Restrições ─────────────────────────────────────────────────────────

        # 1) Balanço de potência em cada hora
        #    Pgrid_buy + PV + Pbess_discharge = Demanda + Pbess_charge + Pgrid_sell
        def power_balance_rule(model, t):
            return (+ model.Pgrid_buy[t]
                    + model.P_pv[t]
                    + model.Pbess_discharge[t]
                    ==
                    + model.P_demand[t]
                    + model.Pbess_charge[t]
                    + model.Pgrid_sell[t])

        model.power = pyo.Constraint(model.T, rule=power_balance_rule)

        # 2) Dinâmica da bateria hora a hora
        #    E[t] = E[t-1] + eff*delta*Pch[t] - delta*Pdis[t]/eff - beta*delta*E[t]
        def bess_energy_rule(model, t):
            eff  = self.bess.eff
            beta = self.bess.self_discharge

            charge_term    = eff  * delta * model.Pbess_charge[t]
            discharge_term = delta * model.Pbess_discharge[t] / eff
            loss_term      = beta * delta * model.E_bess[t]

            if t == 0:
                E_prev = self.bess.initial_capacity  # hora 0: sem hora anterior
            else:
                E_prev = model.E_bess[t - 1]

            return model.E_bess[t] == E_prev + charge_term - discharge_term - loss_term

        model.bess_energy = pyo.Constraint(model.T, rule=bess_energy_rule)

        # 3) Bateria só pode carregar OU descarregar (não os dois ao mesmo tempo)
    
        def bess_discharge_limit(model, t): # (só descarrega se state=1)
            return model.Pbess_discharge[t] <= self.bess.Pmax * model.state[t] 

        def bess_charge_limit(model, t): # (só carrega se state=0)
            return model.Pbess_charge[t] <= self.bess.Pmax * (1 - model.state[t])

        model.bess_dis_limit = pyo.Constraint(model.T, rule=bess_discharge_limit)
        model.bess_ch_limit  = pyo.Constraint(model.T, rule=bess_charge_limit)

        # ── Função objetivo ────────────────────────────────────────────────────
        # Minimizar custo total de compra de energia da rede
        def objective_rule(model):
            return delta * sum( model.tariff[t] * model.Pgrid_buy[t]
                                - model.tariff[t] * model.Pgrid_sell[t]
                                for t in model.T)

        model.funcao_objetivo = pyo.Objective(rule=objective_rule,
                                              sense=pyo.minimize)
        self.model = model

    def solve(self):
        solver   = SolverFactory('highs')
        solution = solver.solve(self.model, tee=True)  # tee=True imprime o log

        # Verifica se resolveu com sucesso
        if (solution.solver.status == SolverStatus.ok and
            solution.solver.termination_condition == TerminationCondition.optimal):
            print("\n✓ Solução ótima encontrada!")
            print(f"  Custo total: R$ {pyo.value(self.model.funcao_objetivo):.4f}")
            
            # Exibir valores das variáveis
            print("\n" + "="*80)
            print("VALORES DAS VARIÁVEIS")
            print("="*80)
            
            results = []
            for t in self.model.T:
                results.append({
                    'Hora': t,
                    'P_Grid_Buy (kW)': f"{pyo.value(self.model.Pgrid_buy[t]):.4f}",
                    'P_Grid_Sell (kW)': f"{pyo.value(self.model.Pgrid_sell[t]):.4f}",
                    'P_BESS_Charge (kW)': f"{pyo.value(self.model.Pbess_charge[t]):.4f}",
                    'P_BESS_Discharge (kW)': f"{pyo.value(self.model.Pbess_discharge[t]):.4f}",
                    'E_BESS (kWh)': f"{pyo.value(self.model.E_bess[t]):.4f}"
                })
            
            results_df = pd.DataFrame(results)
            print(results_df.to_string(index=False))


# ── Execução ───────────────────────────────────────────────────────────────────
smarthome = SmartHome(parameters, P_demand_data, P_pv_data, tariff_data)
smarthome.build()
smarthome.solve()
